In [ ]:
# fine tune it up
!pip install -Uq 'git+https://github.com/huggingface/transformers.git'
!pip install -Uq accelerate peft datasets trl

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM 
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from huggingface_hub import login

login(token="****")

In [ ]:
model_id = "CohereLabs/c4ai-command-r7b-arabic-02-2025"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
dataset = load_dataset("Shams03/Ara-Egy-Medical-QA", split="train")

split_dataset = dataset.train_test_split(test_size=5000, seed=42)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]

print(f"Training examples: {len(train_data)}")
print(f"Testing examples: {len(eval_data)}")

In [ ]:
system_preamble = "أنت مساعد طبي دقيق ومتميز. قدم معلومات طبية واضحة وموجزة ومفيدة."

def format_chat(example):
    messages = [
        {"role": "system", "content": system_preamble},
        {"role": "user", "content": example["Question"]},
        {"role": "assistant", "content": example["Answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_data = train_data.map(format_chat)
eval_data = eval_data.map(format_chat)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", 
    dtype=torch.float16 
)

model.config.use_cache = False

In [ ]:
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.07, 
    bias="none", 
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

training_args = SFTConfig(
    output_dir="./cohere-medical-qa-results",
    num_train_epochs=1,             
    per_device_train_batch_size=3,  
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  
    gradient_checkpointing=True,   
    optim="adamw_torch",           
    learning_rate=2e-4,
    fp16=True,                     
    
    logging_strategy="steps",
    logging_steps=50,               
    logging_first_step=True,      
    disable_tqdm=True,           
    
    eval_strategy="epoch",         
    save_strategy="epoch",
    max_grad_norm=0.3,
    warmup_steps=10,               
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",      
    max_length=400
)

trainer = SFTTrainer(
    model=model,                    
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,        
    processing_class=tokenizer      
)

In [ ]:
print("شغال هالحين")
trainer.train()
print("خلص")


trainer.model.save_pretrained("ArEgyMedical-chatbot-adapter-16bit")
tokenizer.save_pretrained("ArEgyMedical-chatbot-adapter-16bit")

In [ ]:
sample_question = eval_data[0]["Question"]

system_preamble = "أنت مساعد طبي دقيق ومتميز. قدم معلومات طبية واضحة وموجزة ومفيدة."

messages = [
    {"role": "system", "content": system_preamble},
    {"role": "user", "content": sample_question}
]


input_ids = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    add_generation_prompt=True, 
    return_tensors="pt"
).to("cuda")

gen_tokens = model.generate(
    input_ids, 
    max_new_tokens=500, 
    do_sample=True, 
    temperature=0.3,
)

gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
print("Question:", sample_question)
print("\nGenerated Answer:\n", gen_text)